# MSD Decoder Workflow API

This notebook shows the compact workflow API for building MSD kernels, compiling
tasks, training MLD/MLE decoders, sampling evaluation data, evaluating curves,
and plotting results.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / "demo" / "msd_utils").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not locate repo root containing demo/msd_utils.")

LOCAL_DECODER_SRC_CANDIDATES = [
    Path.cwd() / ".." / "bloqade-decoders" / "src",
    Path.cwd() / "bloqade-decoders" / "src",
    Path.cwd().parent / "bloqade-decoders" / "src",
    Path.cwd().parent.parent / "bloqade-decoders" / "src",
]
for candidate in LOCAL_DECODER_SRC_CANDIDATES:
    candidate = candidate.resolve()
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break

from bloqade.decoders import TableDecoder  # noqa: E402
from demo.msd_utils.domain.confidence import ConfidenceGurobiDecoder  # noqa: E402
from bloqade.lanes import GeminiLogicalSimulator  # noqa: E402
# NOTE, mtg: use a class? -- object-oriented methods? -- depends.
# use claude/codex? 
# NOTE, mtg: TaskBundle in the GeminiFrontend?-- make sure things are aligned?
# NOTE, mtg: change things in bundles? -- plural and make it an iterator?
# TODO: add sampling from the DEM directly-- jonathan's class in gemini_benchmarking? -- probably not; 
# can just use stim?
from demo.msd_utils import (  # noqa: E402
    DecoderCurveOptions,
    MSDDecoderWorkflowConfig,
    TableDecoderWithSimplerConfidence,
    build_injected_tomography_kernels,
    build_injected_tomography_tasks,
    build_mle_decoder_suite,
    build_msd_tomography_kernels,
    build_msd_primitives,
    build_msd_tomography_tasks,
    evaluate_decoder_curves,
    evaluate_injected_baseline,
    plot_decoder_curves,
    sample_actual_data,
)

## User Configuration

In [2]:
mld_train_shots = 10_000_000
eval_shots = 1_000_000

ideal_theta = 0.3041 * math.pi
ideal_phi = 0.25 * math.pi
ideal_lam = 0.0

theta_offset = 0.30
phi_offset = 0.0
lam_offset = 0.0

theta = ideal_theta + theta_offset
phi = ideal_phi + phi_offset
lam = ideal_lam + lam_offset

target_bloch_vector = np.ones(3, dtype=np.float64) / np.sqrt(3.0)
decoder_primitive_set = build_msd_primitives(theta, phi, lam)
valid_factory_targets = np.array([[1, 0, 1, 1]], dtype=np.uint8)

config = MSDDecoderWorkflowConfig(
    # TODO, sm: separate the training shots from the MLD config and MLE config.
    mld_train_shots=mld_train_shots,
    # TODO, sm: put different concerns in different sections of the API. For the decoders.
    # TODO, sm: create different objects for separate concerns? -- separate out configs.
    eval_shots=eval_shots,
    target_bloch_vector=target_bloch_vector,
    theta=theta,
    phi=phi,
    lam=lam,
    # TODO, stefan mtg: don't call this a "primitive_set" ?
    decoder_primitive_set=decoder_primitive_set,
    valid_factory_targets=valid_factory_targets,
    num_logical_qubits=5,
    output_qubit=0,
    # Use prefix-prepare special tasks so MLD table-training data stays on a
    # smaller special path. Actual/evaluation data uses CliffT below.
    special_kernel_strategy="compiled_inverse_prefix",
    # Use CliffT for the noisy non-Clifford actual/evaluation sampling path.
    # The tsim detector sampler can hit a normalization-underflow assertion on
    # these circuits.
    sim_type="clifft",
    binary_precision=4,
    log=True,
)

Building MSD primitives...


## Kernels And Tasks

In [3]:
simulator = GeminiLogicalSimulator()

msd_tomography_kernels = build_msd_tomography_kernels(config)
injected_tomography_kernels = build_injected_tomography_kernels(config)

msd_tomography_tasks = build_msd_tomography_tasks(
    simulator,
    config,
    msd_tomography_kernels,
)
injected_tomography_tasks = build_injected_tomography_tasks(
    simulator,
    config,
    injected_tomography_kernels,
)

Building MSD tomography kernels...
Building injected tomography kernels...
Building MSD simulator tasks...
Building injected simulator tasks...


## Decoder Training

In [4]:
mld_decoders = build_mle_decoder_suite(
    msd_tomography_tasks,
    gurobi_decoder_cls=TableDecoderWithSimplerConfidence,
    decoder_init_args={
        "num_shots": config.mld_train_shots,
        "step_size": None,
    },
)

mle_decoders = build_mle_decoder_suite(
    msd_tomography_tasks,
    gurobi_decoder_cls=ConfidenceGurobiDecoder,
)

Building MLE decoder for X...


100%|██████████| 1/1 [00:01<00:00,  1.32s/it]


Building MLE decoder for Y...


100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


Building MLE decoder for Z...


100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

Building MLE decoder for X...
Restricted license - for non-production use only - expires 2027-11-29
Building MLE decoder for Y...
Building MLE decoder for Z...


## Sampling, Curves, And Plot

In [5]:
injected_summary = evaluate_injected_baseline(
    injected_tomography_tasks,
    config,
    # TODO: change code to use TableDecoderWithSimplerConfidence instead.
    table_decoder_cls=TableDecoder,
    raw=False,
)

Evaluating corrected injected baseline...


In [6]:
injected_summary

{'point': 0.9656607049143543,
 'median': 0.9656607049143543,
 'low': 0.9648500964552595,
 'high': 0.966471313373449,
 'error': 0.0008106084590947829,
 'bloch': (0.654964, 0.651396, 0.306736)}

In [ ]:
# NOTE, mtg: add a log for when MLE decoder starts or ends
# NOTE, mtg: can show a progress bar?
# NOTE, mtg: if something runs -- takes 6 minutes -- can split it into extra blocks in the notebook?
actual_data = sample_actual_data(msd_tomography_tasks, config)

curves = evaluate_decoder_curves(
    actual_data,
    {
        "MLD": mld_decoders,
        "MLE": mle_decoders,
    },
    config,
    curve_options=DecoderCurveOptions(
        threshold_points=24,
        threshold_policy="quantile",
        selection_mode="threshold",
    ),
)

# TODO: subset the curves so that things under the "min_accepted_fraction" just aren't plotted at all
# on the curve
fig, ax = plot_decoder_curves(
    curves,
    injected_summary=injected_summary,
    title="MSD Decoder Postselection Curves",
)
fig

Sampling actual data for X with 1,000,000 shots...
Sampling actual data for Y with 1,000,000 shots...
Sampling actual data for Z with 1,000,000 shots...
Evaluating MLD curve...
Evaluating MLE curve...


MLE X: factory decode:   0%|          | 0/1000000 [00:00<?, ?shots/s]

MLE Y: factory decode:   0%|          | 0/1000000 [00:00<?, ?shots/s]

MLE Z: factory decode:   0%|          | 0/1000000 [00:00<?, ?shots/s]

Restricted license - for non-production use only - expires 2027-11-29
